In [14]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import numpy as np


In [37]:
def load_period_data(period_id, folder="../data/processed"):
    train_path = f"{folder}/snn_train_period_{period_id}.csv"
    test_path  = f"{folder}/snn_test_period_{period_id}.csv"
    
    df_train = pd.read_csv(train_path)
    df_test  = pd.read_csv(test_path)
    
    return df_train, df_test


def filter_by_stock(df, stock_ticker):
    return df[df['ticker'] == stock_ticker].reset_index(drop=True)


def load_and_filter(period_id, stock_ticker=None, folder="../data/processed"):
    df_train, df_test = load_period_data(period_id, folder)
    
    #if stock_ticker is not None:
    #    df_train = filter_by_stock(df_train, stock_ticker)
    #    df_test  = filter_by_stock(df_test, stock_ticker)
    
    return df_train, df_test


def prepare_X_y(df, target_col="target"):
    X = df.drop(columns=[target_col, 'ticker', 'date'], errors='ignore')
    y = df[target_col] + 1  # -1,0,1 → 0,1,2
    y_cat = to_categorical(y, num_classes=3)
    
    return X.values, y_cat, y.values


def prepare_data(df_train, df_test, target_col="target"):
    X_train, y_train_cat, y_train = prepare_X_y(df_train, target_col)
    X_test, y_test_cat, y_test = prepare_X_y(df_test, target_col)

    return X_train, y_train_cat, y_train, X_test, y_test_cat, y_test


def build_shallow_nn(input_dim, learning_rate=0.001):
    model = Sequential([
        Dense(31, activation='relu', input_dim=input_dim),
        Dense(31, activation='relu'),
        Dense(3, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


def train_model(X_train, y_train_cat, X_val, y_val_cat, epochs=20, batch_size=32):
    input_dim = X_train.shape[1]
    
    model = build_shallow_nn(input_dim=input_dim)
    
    model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=epochs,
        batch_size=batch_size,
        verbose=2
    )
    
    return model



def predict_model(model, X_test):
    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    return y_pred, y_pred_probs


def evaluate_metrics(y_test, y_pred, y_pred_probs):
    print("\n=== Confusion Matrix ===")
    print(confusion_matrix(y_test, y_pred))
    
    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=['SELL','HOLD','BUY']))
    
    try:
        auc = roc_auc_score(
            to_categorical(y_test, num_classes=3),
            y_pred_probs,
            multi_class='ovr'
        )
        print(f"\nAUC (one-vs-rest): {auc:.4f}")
    except Exception as e:
        print("Error calculando AUC:", e)


def run_shallow_nn(period_id, epochs=20, batch_size=32):
    
    # Load
    df_train, df_test = load_period_data(period_id)
    
    # Prepare
    X_train, y_train_cat, y_train, X_test, y_test_cat, y_test = prepare_data(df_train, df_test)
    
    # Train (⚠️ usando test como validation, solo para debug)
    model = train_model(
        X_train, y_train_cat,
        X_test, y_test_cat,
        epochs=epochs,
        batch_size=batch_size
    )
    
    # Predict
    y_pred, y_pred_probs = predict_model(model, X_test)
    
    # Evaluate
    evaluate_metrics(y_test, y_pred, y_pred_probs)
    
    return model



In [ ]:
period_id = 0
model = run_shallow_nn(period_id,  epochs=100)

Epoch 1/100


/Users/sergio/Documents/NeuralNetworks/NeuralNetworks-Group1/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9724/9724 - 3s - 330us/step - accuracy: 0.3993 - loss: 1.0811 - val_accuracy: 0.4729 - val_loss: 1.0630
Epoch 2/100
9724/9724 - 3s - 293us/step - accuracy: 0.4090 - loss: 1.0743 - val_accuracy: 0.4667 - val_loss: 1.0709
Epoch 3/100
9724/9724 - 3s - 300us/step - accuracy: 0.4176 - loss: 1.0697 - val_accuracy: 0.4492 - val_loss: 1.0802
Epoch 4/100
9724/9724 - 3s - 292us/step - accuracy: 0.4219 - loss: 1.0669 - val_accuracy: 0.4826 - val_loss: 1.0535
Epoch 5/100
9724/9724 - 3s - 292us/step - accuracy: 0.4238 - loss: 1.0655 - val_accuracy: 0.4386 - val_loss: 1.0814
Epoch 6/100
9724/9724 - 3s - 293us/step - accuracy: 0.4244 - loss: 1.0647 - val_accuracy: 0.4857 - val_loss: 1.0576
Epoch 7/100
9724/9724 - 3s - 293us/step - accuracy: 0.4263 - loss: 1.0636 - val_accuracy: 0.4896 - val_loss: 1.0370
Epoch 8/100
9724/9724 - 3s - 293us/step - accuracy: 0.4261 - loss: 1.0629 - val_accuracy: 0.4640 - val_loss: 1.0594
Epoch 9/100
9724/9724 - 3s - 293us/step - accuracy: 0.4278 - loss: 1.0621 - val_accu